# Lab 0 – Umgebungs-Check und Baseline-Pipeline

**Ziel (15 Min):** Alle haben eine laufende Umgebung, kennen den Korpus und haben eine *naive* RAG-Pipeline
gesehen, die wir in den folgenden Labs Schritt für Schritt verbessern.

**Ablauf**
1. Teil A – Walkthrough (Trainer führt vor, alle führen mit aus)
2. Teil B – Aufgaben
3. Teil C – Debrief-Fragen

> Alle Labs nutzen das Paket `ragkurs` im Repo. Der Code ist bewusst kurz und lesbar – bei Fragen einfach
> `??` an eine Funktion hängen, z. B. `chunk_fixed??`.

## Teil A – Walkthrough

### A1 Umgebung prüfen

In [ ]:
import sys, os
_root = os.getcwd()                      # Repo-Root finden (Notebook liegt in labs/ oder labs/solutions/)
while not os.path.isdir(os.path.join(_root, "ragkurs")):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root); os.chdir(_root)  # Repo-Root in den Pfad (Notebook liegt in labs/)

from ragkurs import settings
print("Chat-Modell :", settings.chat_model)
print("Judge-Modell:", settings.judge_model)
print("Embeddings  :", settings.embed_model, f"({settings.embed_dim} Dim.)")
print("API-Key     :", "vorhanden" if settings.openai_api_key else "FEHLT -> .env prüfen")
print("Offline-Modus (FAKE_EMBEDDINGS):", settings.fake_embeddings)

### API-Spickzettel für dieses Lab

Alle Funktionen mit Signatur und Beispiel: `docs/CHEATSHEET.md`. Im Notebook: `funktion??` zeigt den Quelltext.

```python
from ragkurs import load_corpus, chunk_fixed, HybridIndex, RAGPipeline, PipelineConfig
from ragkurs.eval import load_golden, evaluate_retrieval, retrieval_summary

docs   = load_corpus()                                   # 42 Documents; load_corpus(pdf_parser="docling") ab Lab 4
chunks = chunk_fixed(docs, chunk_size=800, chunk_overlap=100)
index  = HybridIndex(collection="baseline").build(chunks)
pipe   = RAGPipeline(index, PipelineConfig(retrieval="dense", k=5, name="baseline"))

result = pipe.run("Frage?", user_roles=["employee"])    # Retrieval + Antwort;  result.show()  zeigt alles
result.answer.text            # Antworttext          result.retrieved_doc_ids   # ['hr-urlaubsrichtlinie', ...]
result.trace["t_total_ms"]    # Zeiten, Tokens, Kosten, Kandidaten

golden = load_golden()                                   # 64 Fragen; load_golden(types=["tabelle"]) / ids=[...] / answerable=False
item = next(g for g in golden if g["id"] == "g33")       # eine bestimmte Frage
df  = evaluate_retrieval(pipe, golden, user_roles=["employee"])   # nur Retrieval, kein LLM -> DataFrame (hit, recall, first_rank, ...)
retrieval_summary(df)                                    # {'hit_rate': .., 'recall': .., 'mrr': .., 'avg_ms': ..}
df.groupby("type")["hit"].mean()                         # Hit-Rate je Fragetyp
```

### A2 Der Korpus: „Aurelia Maschinenbau GmbH“

42 Dokumente eines fiktiven Maschinenbauers: HR-Richtlinien, IT-Richtlinien, Produkthandbücher, Preisliste,
Compliance – teils Markdown, teils **nur als PDF** (mit Tabellen). Dazu zwei Gesetzesauszüge (BUrlG, ArbZG).

Eingebaute Fallen, die produktive Systeme genauso haben:
- **Near-Misses**: AX-100 / AX-200 / AX-300 / BX-500 (ähnliche Handbücher, andere Zahlen), Karlsruhe vs. Linz, Verwaltung vs. Produktion, Neu- vs. Gebrauchtmaschinen, EU- vs. Export-Lieferung
- **Stale Data**: ersetzte Fassungen (Reisekosten 2024/2025, IT-Sicherheit 2022, Preisliste 2025, Wartungsplan 2024, AnyConnect) neben den gültigen
- **Zugriffsrechte**: Gehaltsbänder (nur HR/Management), Rabattrichtlinie (nur Vertrieb)
- **Multi-Hop**: Antworten, die zwei Dokumente brauchen (Richtlinie + Gesetz)
- **Tabellen**: Fristen, Preise, Fehlercodes stehen in Tabellen – in PDFs geht die Struktur beim naiven Parsen verloren

In [ ]:
from ragkurs import load_corpus
from ragkurs.loading import corpus_overview

docs = load_corpus()            # Standard: PDFs mit pypdf (schnell, aber Tabellen zerfallen)
corpus_overview(docs)

In [ ]:
# So sieht ein PDF nach pypdf aus - Tabellenzellen stehen untereinander, die Zuordnung ist weg:
pdf_doc = next(d for d in docs if d.doc_id == "hr-reisekosten-2026")
print(pdf_doc.text[600:1300])

### A3 Das Golden Set

64 Fragen mit Referenzantwort, Quelldokument(en) und Typ. Damit messen wir in jedem Lab, ob eine Änderung
wirklich hilft – statt „sieht gut aus“ (LGTM@few). Die zwei `acl`-Fragen (Gehaltsbänder, Rabatte) sind für die
Rolle `employee` absichtlich unauffindbar – sie werden in Lab 6 gesondert geprüft und in den Retrieval-Kennzahlen
übersprungen.

In [ ]:
from ragkurs.eval import load_golden
import pandas as pd

golden = load_golden()
pd.DataFrame(golden)["type"].value_counts()

In [ ]:
pd.DataFrame(golden)[["id", "type", "question", "source_docs"]].head(8)

### A4 Die naive Baseline

Fixed-Size-Chunks (800 Zeichen) → Dense-Embeddings → Top-5 → Prompt → Antwort. Genau das, was die meisten
Teams als ersten Prototyp bauen – inklusive des typischen Fehlers: **alles aus dem Ordner in den Index**, auch die
ersetzten Fassungen (`status_filter=None`). Versionen, Rechte und Metadaten kommen erst später ins Spiel.

In [ ]:
from ragkurs import chunk_fixed, HybridIndex, RAGPipeline, PipelineConfig

chunks = chunk_fixed(docs, chunk_size=800, chunk_overlap=100)
print(len(chunks), "Chunks")
print(chunks[10])
print(chunks[10].text[:300])

In [ ]:
index = HybridIndex(collection="baseline").build(chunks)   # Qdrant im Prozess, kein Server
index.stats

In [ ]:
baseline = RAGPipeline(index, PipelineConfig(retrieval="dense", k=5, status_filter=None, name="baseline"))  # naiv: kein Versionsfilter
result = baseline.run("Wie viele Urlaubstage haben Vollzeitmitarbeitende pro Jahr?", user_roles=["employee"])
result.show()

`result.trace` enthält alles, was wir später fürs Debugging brauchen: Kandidaten mit Scores, Zeiten pro Stufe,
Tokens, Kosten.

In [ ]:
{k: v for k, v in result.trace.items() if k != "candidates"}

## Teil B – Aufgaben

### B1 Drei Fragen selbst stellen
Stellt der Baseline drei Fragen aus dem Golden Set – eine `faktisch`, eine `near-miss`, eine `negativ` –
und notiert: Ist die Antwort korrekt? Stand die richtige Quelle im Kontext?

In [ ]:
# === LOESUNG START ===
# HINWEIS: golden ist eine Liste von Dicts; result.retrieved_doc_ids zeigt die Dokumente im Kontext.
for typ in ("faktisch", "near-miss", "negativ"):
    item = next(g for g in golden if g["type"] == typ)
    r = baseline.run(item["question"], user_roles=["employee"])  # Pipeline mit der Frage aufrufen (Rolle employee)
    print(f"\n=== {typ} | {item['id']} ===")
    print("Frage    :", item["question"])
    print("Antwort  :", r.answer.text)
    print("Referenz :", item["ground_truth"])
    print("Quellen im Kontext:", r.retrieved_doc_ids, "| erwartet:", item["source_docs"])  # gefundene vs. erwartete Dokumente ausgeben
# === LOESUNG ENDE ===

### B2 Retrieval-Trefferquote der Baseline messen
Nutzt `evaluate_retrieval` (kein LLM-Call, nur Retrieval) und `retrieval_summary`. Wie hoch ist die Hit-Rate@5 –
und wie hoch **p@1** (richtige Quelle auf Platz 1)? Die Differenz ist der Spielraum für Reranking (Lab 3).

In [ ]:
from ragkurs.eval import evaluate_retrieval, retrieval_summary

# === LOESUNG START ===
# HINWEIS: evaluate_retrieval(pipeline, golden, user_roles) -> DataFrame; retrieval_summary(df) -> Kennzahlen
df_base = evaluate_retrieval(baseline, golden, user_roles=["employee"])  # Retrieval-Evaluation auf dem Golden Set
retrieval_summary(df_base)  # Kennzahlen zusammenfassen
# === LOESUNG ENDE ===

In [ ]:
# Welche Fragetypen scheitern am häufigsten?
df_base.groupby("type")["hit"].mean().sort_values()

## Teil C – Debrief

1. Die Hit-Rate@5 ist bei 42 Dokumenten hoch – die naive Baseline wirkt „fertig“. Woran merkt man trotzdem,
   dass sie es nicht ist? (p@1 vs. Hit@5, ersetzte Fassungen im Index, Rechte, Kosten, 4.000 statt 42 Dokumente)
2. Was fehlt dieser Pipeline, um produktionsreif zu sein? (Sammelt Stichworte – das ist die Agenda der zwei Tage.)

**Merksatz:** Ohne Golden Set ist jede Verbesserung eine Vermutung.